In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "04-inference-engine/flash-attention")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# FlashAttention — practice

Five exercises. Each one is a function with the body removed and a `check()` call
underneath it. Fill in the body (delete its `raise NotImplementedError` line), run the
cell: every check prints `[PASS]` or stops the cell with an `AssertionError` that shows
yours against the expected value. Until you attempt an exercise, running the notebook
stops there with `NotImplementedError`.

Do them in order — each builds on the last. Exercise 3 is the one that matters;
if you only do one, do that one.

Worked solutions are in `flash_attention_minimal.py`. Try not to look until you've
been stuck for ten minutes.

## Setup — run this first

In [ ]:
import numpy as np

np.random.seed(0)
np.set_printoptions(precision=4, suppress=True)

# --- fixtures ---------------------------------------------------------
N, d = 8, 4
Q = np.random.randn(N, d)
K = np.random.randn(N, d)
V = np.random.randn(N, d)
scale = 1.0 / np.sqrt(d)

# --- reference implementation, used only to grade you -----------------
def _reference(Q, K, V, causal=False):
    S = Q @ K.T / np.sqrt(Q.shape[-1])
    if causal:
        S = np.where(np.arange(S.shape[1])[None, :] <= np.arange(S.shape[0])[:, None],
                     S, -np.inf)
    P = np.exp(S - S.max(axis=-1, keepdims=True))
    return (P / P.sum(axis=-1, keepdims=True)) @ V

REF = _reference(Q, K, V)
REF_CAUSAL = _reference(Q, K, V, causal=True)

# --- grader -----------------------------------------------------------
def check(name, got, want, tol=1e-9):
    '''Print [PASS], or stop the cell with an AssertionError.'''
    if got is None:
        raise NotImplementedError(f"{name}: returned None — is the body still a TODO?")
    got, want = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    assert got.shape == want.shape, f"[FAIL] {name}: shape {got.shape}, expected {want.shape}"
    diff = np.abs(got - want).max() if got.size else 0.0
    assert diff < tol, (f"[FAIL] {name}   max diff = {diff:.2e}\n"
                        f"      yours    : {np.ravel(got)[:4]}\n      expected : {np.ravel(want)[:4]}")
    print(f"[PASS] {name}   max diff = {diff:.2e}")

print("setup ok — Q, K, V are", Q.shape)

## Exercise 1 — naive attention

Warm-up. Build the version FlashAttention replaces, so you have something to
compare against and can see the `N x N` intermediate with your own eyes.

    S = Q Kᵀ / √d          scores
    P = softmax(S)         row-wise, and remember to subtract the row max
    O = P V

In [ ]:
def naive_attention(Q, K, V):
    d = Q.shape[-1]
    # TODO: three lines
    #   1. S = scaled scores          -> (N, N)
    #   2. P = row-wise stable softmax of S
    #   3. return P @ V
    # YOUR CODE HERE: then delete the raise line
    raise NotImplementedError("Exercise 1: naive_attention")


check("naive_attention", naive_attention(Q, K, V), REF)

## Exercise 2 — online softmax statistics

Stream over a vector in blocks and maintain two running numbers:

| | |
|---|---|
| `m` | largest value seen so far |
| `l` | running sum of `exp(x - m)` |

When a block arrives with a bigger max, `l` was accumulated against a stale `m`
and is wrong by a known factor. Fix it before adding the new block:

    m_new = max(m, block.max())
    alpha = exp(m - m_new)
    l     = alpha * l + sum(exp(block - m_new))

Never hold the whole vector.

In [ ]:
def online_softmax_stats(x, block_size):
    '''Return (m, l) matching x.max() and exp(x - x.max()).sum(),
    computed in one streaming pass over blocks.'''
    m = -np.inf
    l = 0.0
    for start in range(0, len(x), block_size):
        block = x[start:start + block_size]
        # TODO: update m and l using the correction factor
        # YOUR CODE HERE: then delete the raise line
        raise NotImplementedError("Exercise 2: online_softmax_stats")
    return m, l


x = np.random.randn(100) * 3
m_got, l_got = online_softmax_stats(x, block_size=7)
check("online m", m_got, x.max())
check("online l", l_got, np.exp(x - x.max()).sum())

## Exercise 3 — merging partial attention  ⭐

**This is the whole idea.** Everything else is loop plumbing.

`attention_chunk` is given: it attends one query against *some* of the keys and
returns the result **unnormalized**, plus that chunk's `m` and `l`. Your job is
`merge` — combine two such partial results as if they'd been computed together.

Rescale both sides to a shared max, then add:

    m = max(m1, m2)
    a1, a2 = exp(m1 - m), exp(m2 - m)
    o = a1*o1 + a2*o2
    l = a1*l1 + a2*l2

If this works, partial attention results are composable — which is why you can
tile across SRAM, shard across GPUs, or split a KV cache and recombine.

In [ ]:
def attention_chunk(q, K_chunk, V_chunk):
    '''GIVEN. Unnormalized attention of one query against a chunk of K/V.'''
    s = q @ K_chunk.T / np.sqrt(q.shape[-1])
    m = s.max()
    p = np.exp(s - m)
    return p @ V_chunk, m, p.sum()


def merge(o1, m1, l1, o2, m2, l2):
    '''Combine two partial results into one. Returns (o, m, l), still unnormalized.'''
    # TODO: rescale both to a shared max, then add
    # YOUR CODE HERE: then delete the raise line
    raise NotImplementedError("Exercise 3: merge")


q = Q[0]
o1, m1, l1 = attention_chunk(q, K[:3], V[:3])      # first 3 keys
o2, m2, l2 = attention_chunk(q, K[3:], V[3:])      # the rest
o, m, l = merge(o1, m1, l1, o2, m2, l2)

check("merge", None if o is None else o / l, REF[0])

# should also hold in the other order, and across an uneven 3-way split
o_r, _, l_r = merge(o2, m2, l2, o1, m1, l1)
check("merge (reversed)", o_r / l_r, REF[0])
a = attention_chunk(q, K[:1], V[:1])
b = attention_chunk(q, K[1:6], V[1:6])
c = attention_chunk(q, K[6:], V[6:])
ab = merge(*a, *b)
abc = merge(*ab, *c)
check("merge (3-way)", abc[0] / abc[2], REF[0])

## Exercise 4 — the full tiled forward pass

Now vectorise Exercise 3 over a *block* of queries and fold the merge into the
inner loop. Per query block, carry `Oi` `(bq, d)`, `mi` `(bq,)`, `li` `(bq,)`.

For each key/value block:

    Sij   = (Qi @ Kj.T) * scale                 # (bq, bkv) — the only tile you hold
    m_new = maximum(mi, Sij.max(axis=1))
    alpha = exp(mi - m_new)                     # (bq,)
    Pij   = exp(Sij - m_new[:, None])
    li    = alpha * li + Pij.sum(axis=1)
    Oi    = alpha[:, None] * Oi + Pij @ Vj
    mi    = m_new

Normalize **once**, after the inner loop: `O_block = Oi / li[:, None]`.

Watch the broadcasting — `alpha` is per-row, so it needs `[:, None]` against `Oi`
but not against `li`.

In [ ]:
def flash_attention(Q, K, V, block_q=4, block_kv=4):
    N, d = Q.shape
    scale = 1.0 / np.sqrt(d)
    O = np.zeros((N, d))
    L = np.zeros(N)                     # logsumexp per row = m + log(l)

    for i in range(0, N, block_q):
        Qi = Q[i:i + block_q]
        bq = Qi.shape[0]

        Oi = np.zeros((bq, d))
        mi = np.full(bq, -np.inf)
        li = np.zeros(bq)

        for j in range(0, N, block_kv):
            Kj, Vj = K[j:j + block_kv], V[j:j + block_kv]
            # TODO: one merge step — update mi, li, Oi
            # YOUR CODE HERE: then delete the raise line
            raise NotImplementedError("Exercise 4: flash_attention")

        # TODO: normalize this query block into O, and store m + log(l) into L
        pass

    return O, L


O_got, L_got = flash_attention(Q, K, V, block_q=4, block_kv=4)
check("flash_attention", O_got, REF)

S_full = Q @ K.T * scale                 # the logsumexp you should have stored in L
check("logsumexp L", L_got, S_full.max(axis=1) + np.log(np.exp(S_full - S_full.max(axis=1, keepdims=True)).sum(axis=1)))

# the real test: the answer must not depend on tile size
worst = max(np.abs(flash_attention(Q, K, V, bq, bkv)[0] - REF).max()
            for bq in (1, 2, 3, 5, 8) for bkv in (1, 2, 3, 5, 8))
assert worst < 1e-9, f"[FAIL] tile-size independence   worst diff over 25 configs = {worst:.2e}"
print(f"[PASS] tile-size independence   worst diff over 25 configs = {worst:.2e}")

## Exercise 5 — causal masking and tile skipping (stretch)

A token may not attend to the future. Two things to add:

1. **Mask the diagonal tile.** For a tile at query offset `i`, key offset `j`,
   set entries where `col > row` to `-inf` before taking the max. This is what
   makes the answer correct.
2. **Skip tiles entirely.** If the whole tile sits above the diagonal
   (`j > i + bq - 1`) there is nothing to compute — `continue`. This is what
   makes it fast: roughly half the tiles are never touched, which is where the
   ~2x causal speedup comes from in real kernels.

Step 1 alone gives the right output but computes every tile, so the check also
counts the tiles you compute (the given `tiles += 1`, which must come after your
skip) and compares that with the tiles that hold at least one visible key.

**A trap this loop never meets, and why.** If a row's running max is still `-inf`
when it meets a tile that is fully masked for that row, `m_new` is `-inf` too, and
both `exp(Sij - m_new)` and `alpha` become `exp(-inf - -inf)` = `nan` (the check
prints it). Here the key loop walks forward from key 0, which every query may see,
so every row's first tile holds a finite score and `mi` is finite from then on; a
later tile that is fully masked for a row only adds `exp(-inf - finite) = 0`. That
holds with or without tile skipping: skipping saves work, not this. A kernel that
walks backwards from the diagonal, uses a sliding window, or aligns the causal mask
bottom-right with more queries than keys can meet a fully masked first tile, and
needs an explicit guard (deep dive sections 2.5 and 11.2:
[`flash-attention-deep-dive.md`](flash-attention-deep-dive.md)).

In [ ]:
def flash_attention_causal(Q, K, V, block_q=2, block_kv=2):
    '''Returns (O, tiles): the causal output and how many K/V tiles were computed.'''
    N, d = Q.shape
    scale = 1.0 / np.sqrt(d)
    O = np.zeros((N, d))
    tiles = 0                           # GIVEN: counts the tiles you compute

    for i in range(0, N, block_q):
        Qi = Q[i:i + block_q]
        bq = Qi.shape[0]
        Oi, mi, li = np.zeros((bq, d)), np.full(bq, -np.inf), np.zeros(bq)

        for j in range(0, N, block_kv):
            # YOUR CODE HERE: fill in the TODOs below, then delete the raise line
            raise NotImplementedError("Exercise 5: flash_attention_causal")
            # TODO: skip this tile if it is entirely above the diagonal
            tiles += 1                  # GIVEN: keep this after your skip
            Kj, Vj = K[j:j + block_kv], V[j:j + block_kv]
            Sij = (Qi @ Kj.T) * scale
            # TODO: mask entries where the key index exceeds the query index
            #       rows = np.arange(i, i + bq)[:, None]
            #       cols = np.arange(j, j + Kj.shape[0])[None, :]
            # TODO: the same merge step as Exercise 4

        # TODO: normalize
        pass

    return O, tiles


O_got, tiles_got = flash_attention_causal(Q, K, V)
check("causal", O_got, REF_CAUSAL)

def _visible_tiles(n, bq, bkv):
    '''Tiles that hold at least one key a query may see (col <= row).'''
    return sum(bool(np.any(np.arange(j, min(j + bkv, n))[None, :] <= np.arange(i, min(i + bq, n))[:, None]))
               for i in range(0, n, bq) for j in range(0, n, bkv))

for bq, bkv in ((2, 2), (1, 1), (3, 2), (2, 3), (4, 1), (5, 8), (8, 8)):
    O_t, t = flash_attention_causal(Q, K, V, bq, bkv)
    check(f"causal, {bq}x{bkv} tiles", O_t, REF_CAUSAL)
    want, total = _visible_tiles(N, bq, bkv), -(-N // bq) * -(-N // bkv)
    assert t == want, (f"[FAIL] tile skipping, {bq}x{bkv} tiles: you computed {t} of {total} tiles; "
                       f"only {want} hold a visible key -> skip the tiles above the diagonal")
print(f"[PASS] tile skipping   {tiles_got} of {(-(-N // 2)) ** 2} tiles computed at 2x2 (and the same check at 6 more tile shapes)")

# The trap from the text, in one row: a running max of -inf meets a fully masked tile.
with np.errstate(invalid="ignore"):
    _row, _m = np.full(2, -np.inf), -np.inf
    _m_new = max(_m, _row.max())
    print("fully masked first tile: exp(Sij - m_new) =", np.exp(_row - _m_new), " alpha =", np.exp(_m - _m_new))

---

## If you want to keep going

Roughly in order of how much they'll teach you per hour:

- **Count the memory traffic.** Wrap reads of `K` and `V` in a counter and compare
  bytes moved for naive vs tiled across a few tile sizes. The curve is the roofline
  argument, made concrete.
- **Backward pass.** Recompute `S` and `P` from `Q, K, V` and the saved `L` instead
  of storing `P`. You'll need `D = rowsum(dO * O)`. This is the part that makes the
  memory saving hold during *training*, not just inference.
- **Flash-Decoding.** One query row, long KV cache, split across "devices" and
  merged with your Exercise 3 function. Roughly ten lines once `merge` exists.
- **Port it to Triton** and run it on a GPU (a free Colab or Kaggle T4 is enough, T1;
  `TRITON_INTERPRET=1` runs it on a CPU). That's where tile size stops being arbitrary
  and starts being dictated by SRAM per SM. The deep dive's section 11 walks through one.